[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/hubbard_dimer/hubbard_dimer.ipynb)

# The Hubbard Dimer

A state of two electrons in four orbitals is the wedge of the two occupied ones, `left_up ^ right_down`: it vanishes when both sit in one orbital and flips sign when they trade places. A one-electron hop acts on both electrons by one line, `(single_hopping ^ (Pair & Complement)).trace(1, 3)`, either electron hopping in turn. On the singlet and the state where the electrons share a site, the Hamiltonian less half the repulsion, applied twice, is a number, the square of a rate. That one fact gives the energies, half the repulsion plus or minus the rate; the ground state, `rate * singlet - shifted` with `shifted` the singlet under the shifted Hamiltonian; and the motion in time, a cosine and a sine of the rate. When sharing a site is expensive, the electrons mostly stay apart, and what is left of the hopping is a slow exchange of their spins.

The model is Hubbard's, from *Electron correlations in narrow energy bands* (Proceedings of the Royal Society A 276, 238, 1963), and the slow exchange at strong repulsion, four times the hopping squared over the repulsion, is Anderson's superexchange, from *New approach to the theory of superexchange interactions* (Physical Review 115, 2, 1959). [Carrascal, Ferrer, Smith and Burke](https://arxiv.org/abs/1502.02194) take the dimer much further as a case study for density functional theory, starting from the competition between repulsion and an asymmetry of the sites that section 6 traces. The exchange is the coupling the [two spins](../two_spins/two_spins.ipynb) notebook takes as given, and the [Hubbard ring](../hubbard_ring/hubbard_ring.ipynb) lifts the same hopping, by the same line, to four electrons on four sites.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from numga import Algebra, NumpyContext, stack
from examples.quantum.hubbard_dimer import render

np.set_printoptions(precision=4, suppress=True)

# Four orbital directions: left up, left down, right up, right down.
ga = Algebra("a+b+c+d+")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Orbital = ga.gatype.vector()
Pair = ga.gatype.bivector()
Complement = ga.gatype.antivector()
Hopping = ga.gatype((Orbital, Orbital))                         # Orbital <- Orbital
Hamiltonian = ga.gatype((Pair, Pair))                          # Pair <- Pair

left_up, left_down = mv.a, mv.b
right_up, right_down = mv.c, mv.d
left = stack([left_up, left_down])                            # [spins] Orbital
right = stack([right_up, right_down])                         # [spins] Orbital
configurations = stack([mv.ab, mv.ac, mv.ad, mv.bc, mv.bd, mv.cd])  # [configurations] Pair
doubles = stack([left_up ^ left_down, right_up ^ right_down]) # [sites] Pair

# One entry per regime. Energies use the hopping strength as their unit.
hopping_strength = 1.0
names = ("Delocalized", "Crossover", "Localized spins")
repulsion_ratios = np.array([0.0, 4.0, 20.0])
repulsions = hopping_strength * repulsion_ratios

## 1. Either electron can hop

An orbital is a vector; two occupied orbitals form a bivector. Repeating an orbital gives zero, and exchanging their order changes the sign. The hopping map acts on either wedge factor in turn and adds the two results: its derivation extension to pairs.

In creation-and-annihilation-operator notation the hopping reads as $\hat T=-t\sum_{\sigma}(\hat c^\dagger_{L\sigma}\hat c_{R\sigma}+\hat c^\dagger_{R\sigma}\hat c_{L\sigma})$.


In [ ]:
# Leaving Orbital open makes the inner product a readout on an arbitrary input.
# Each dyad reads an orbital on one site and returns the same spin on the other.
single_hopping = -hopping_strength * (
    left * (right | Orbital) + right * (left | Orbital)
).sum(axis=0)                                                # [] Orbital <- Orbital


def lift(one_electron):
    """A one-electron map acting on either electron of a pair in turn. Pair & Complement leaves two
    inputs open: the join removes one occupied orbital, and wedging its image back in replaces that
    orbital, with its exchange sign. The trace joins the map's input to the complementary slot."""
    return (one_electron ^ (Pair & Complement)).trace(1, 3)   # [] Pair <- Pair


pair_hopping = lift(single_hopping)                           # [] Pair <- Pair
# pair_hopping(first ^ second) == single_hopping(first) ^ second + first ^ single_hopping(second)

singlet = ((left_up ^ right_down) - (left_down ^ right_up)) / np.sqrt(2)
triplets = stack([left_up ^ right_up,
                  ((left_up ^ right_down) + (left_down ^ right_up)) / np.sqrt(2),
                  left_down ^ right_down])                  # [spin_projections] Pair
symmetric_double = doubles.sum(axis=0) / np.sqrt(2)           # [] Pair

# Same-spin hops are blocked; the opposite-spin triplet cancels by exchange signs.
blocked = pair_hopping(triplets)                             # [spin_projections] Pair
# The singlet's two routes reinforce: singlet_image == -2 * hopping_strength * symmetric_double.
singlet_image = pair_hopping(singlet)                         # [] Pair

## 2. Charge motion lowers the singlet

Two opposite spins may share a site, at an energy cost, the repulsion. Hopping takes the singlet to the symmetric double occupancy, the even sum of the two shared-site states, and back; the repulsion lifts only the latter. So the singlet and the symmetric double occupancy make a block of their own, and on it the Hamiltonian less half the repulsion, applied twice, is a number: the square of the block's rate, `sqrt((repulsion / 2)**2 + 4 * hopping_strength**2)`. A map that squares to a number has two values, plus and minus its square root, so the block's energies are half the repulsion plus or minus the rate. The triplets stay at zero, and the lower energy falls below them by the gap `rate - repulsion / 2`, which tends to four times the hopping squared over the repulsion when sharing a site is expensive. The full spectrum agrees, with one more level, the odd combination of the shared-site states, at the repulsion itself.

In matrix notation the Hamiltonian on the singlet and the symmetric double occupancy reads as $\begin{pmatrix}0&-2t\\-2t&U\end{pmatrix}$, with eigenvalues $U/2 \pm \sqrt{(U/2)^2 + 4t^2}$.

In [ ]:
# Each dyad projects onto one shared-site configuration; the sum measures double occupancy.
double_occupancy = (doubles * doubles.reverse().scalar_product(Pair)).sum(axis=0)   # [] Pair <- Pair


def hamiltonian(repulsion):
    """Either electron hops, and sharing a site costs the repulsion."""
    return pair_hopping + repulsion * double_occupancy                              # [...] Pair <- Pair


def block_rate(repulsion):
    """The rate of the singlet block: the shifted Hamiltonian applied twice there is its square."""
    return np.sqrt((repulsion / 2) ** 2 + 4 * hopping_strength**2)


# A sweep of the repulsion, and the shifted Hamiltonian applied twice to the block at each value.
samples = 241
repulsion_sweep = np.linspace(0.0, repulsions[-1], samples)
sweep_hamiltonians = hamiltonian(repulsion_sweep[:, None])                          # [samples, 1] Pair <- Pair
shift = repulsion_sweep[:, None] / 2                                               # [samples, 1]
block = stack([singlet, symmetric_double])                                         # [2] Pair
once = sweep_hamiltonians(block) - shift * block                                   # [samples, 2] Pair
twice = sweep_hamiltonians(once) - shift * once                                    # [samples, 2] Pair
exchange_gap = block_rate(repulsion_sweep) - repulsion_sweep / 2                   # [samples]
# All six levels, from the full Hamiltonian, to set the block among them.
energies = hamiltonian(repulsion_sweep).eigvalsh()                                  # [samples, modes] Scalar
# At large repulsion the gap approaches four times hopping squared over repulsion, singular at zero.
strong_repulsion = repulsion_sweep[1:]
perturbative_gap = 4 * hopping_strength**2 / strong_repulsion
render.draw_spectrum(repulsion_sweep, hopping_strength, energies, exchange_gap, perturbative_gap, strong_repulsion);

In [ ]:
print("shifted Hamiltonian applied twice, less the rate squared:", np.abs((twice - block_rate(repulsion_sweep[:, None]) ** 2 * block).kernel).max())
print("gap from the block, less the gap of the full spectrum:", np.abs(exchange_gap - (energies.to_array()[:, 1] - energies.to_array()[:, 0])).max())

## 3. The ground state, and how often the electrons share a site

On the block, the rate less the shifted Hamiltonian is zero on the upper state and twice the rate on the lower one, so it projects onto the lower: the ground state is the singlet projected by it, and normalized. At zero repulsion the ground state has the electrons on one site half the time. Strong repulsion leaves mostly one electron per site, in a singlet with spin correlation -3/4 in units of ħ²; the small shared-site part carries no local spin and weakens that correlation. The picture shows how likely each configuration is in the ground state of each regime; the readout below it gives how often the electrons share a site, and their spin correlation.

In bra-ket notation the singly occupied singlet reads as $(|\uparrow,\downarrow\rangle-|\downarrow,\uparrow\rangle)/\sqrt{2}$, and the spin correlation as $\langle\mathbf S_L\cdot\mathbf S_R\rangle$.

In [ ]:
def ground(repulsion):
    """The singlet projected onto the lower state of its block, by the rate less the shifted
    Hamiltonian, and normalized."""
    shifted = hamiltonian(repulsion)(singlet) - repulsion / 2 * singlet            # [...] Pair
    lower = block_rate(repulsion) * singlet - shifted                              # [...] Pair
    return lower / lower.scalar_norm_squared().square_root()                      # [...] Pair


grounds = ground(repulsions)                                                       # [cases] Pair
ground_energies = grounds.reverse().scalar_product(hamiltonian(repulsions)(grounds))   # [cases] Scalar
# Pairing with a configuration gives its real amplitude; squaring gives its probability.
ground_probabilities = configurations.reverse().scalar_product(grounds[:, None]).squared()   # [cases, configurations] Scalar
ground_double = grounds.reverse().scalar_product(double_occupancy(grounds))       # [cases] Scalar
# The open Pair is the identity map. Subtract the shared-site projector to keep
# one electron per site: triplets contribute one quarter, the singlet minus three quarters.
singlet_weight = singlet * singlet.reverse().scalar_product(Pair)
spin_correlation = (Pair - double_occupancy - 4 * singlet_weight) / 4
ground_correlation = grounds.reverse().scalar_product(spin_correlation(grounds))  # [cases] Scalar
render.draw_ground_states(names, repulsion_ratios, ground_probabilities);

In [ ]:
print("ground energies:", ground_energies.to_array(), "  half the repulsion less the rate:", repulsions / 2 - block_rate(repulsions))
print("both electrons on one site, per regime:", ground_double.to_array())
print("spin correlation, per regime:", ground_correlation.to_array())

## 4. Watch the spins exchange

Start with spin up on the left and spin down on the right: half singlet, half triplet of zero spin. Time acts on a state through the cosine and the sine of the Hamiltonian times time, so the state after a while is a pair of real states, the cosine and the sine of the Hamiltonian applied to where it started. The triplet, at zero energy, stays put in the cosine part. The singlet turns within its block: the shifted Hamiltonian's cosine and sine act on it as the cosine and sine of the rate, and the half repulsion mixes the two parts by its own cosine and sine. A configuration's probability is the sum of its squared amplitudes in the two parts. Each regime runs for one cycle of its own exchange phase, the gap times time, and a model of two localized spins alone predicts a swap probability of the squared sine of half that phase. Left to right: free hopping swaps the spins through charge motion, with both electrons on one site half the time; the crossover adds fast charge oscillations on top; strong repulsion follows the spin-only model, hopping only briefly to put both electrons on one site and returning them with their spins exchanged.

In bra-ket notation the two parts read as the real part $\cos(H\tau)|\psi\rangle$ and minus the imaginary part $\sin(H\tau)|\psi\rangle$ of $e^{-iH\tau}|\psi\rangle$, the swap probability as $|\langle\downarrow,\uparrow|e^{-iH\tau}|\uparrow,\downarrow\rangle|^2$, and the spin-only prediction as $\sin^2(J\tau/2)$ for the Heisenberg coupling $J\,\mathbf S_L\cdot\mathbf S_R$, with $J \to 4t^2/U$ at strong repulsion.

In [ ]:
def exchange(repulsion, times):
    """Spin up on the left and spin down on the right after the given times: the cosine and the sine
    of the Hamiltonian times time, applied to it."""
    rate = block_rate(repulsion)
    shifted = hamiltonian(repulsion)(singlet) - repulsion / 2 * singlet            # [...] Pair
    # On the singlet the shifted Hamiltonian's cosine and sine are those of the rate.
    along = np.cos(rate * times) * singlet                                         # [...] Pair
    across = np.sin(rate * times) / rate * shifted                                 # [...] Pair
    # The half repulsion mixes the two by its own cosine and sine; the triplet sits in the cosine part.
    half = 0.5 * repulsion * times
    cosine_part = (triplets[1] + np.cos(half) * along - np.sin(half) * across) / np.sqrt(2)   # [...] Pair
    sine_part = (np.sin(half) * along + np.cos(half) * across) / np.sqrt(2)                  # [...] Pair
    return cosine_part, sine_part


# One cycle of the exchange phase per regime, on that regime's own timescale.
steps = 1200
exchange_phase = np.linspace(0.0, 2 * np.pi, steps + 1)
gaps = block_rate(repulsions) - repulsions / 2                                     # [cases]
times = exchange_phase / gaps[:, None]                                             # [cases, times]
cosine_part, sine_part = exchange(repulsions[:, None], times)                      # [cases, times] Pair each
cosine_amplitudes = configurations.reverse().scalar_product(cosine_part[..., None])   # [cases, times, configurations] Scalar
sine_amplitudes = configurations.reverse().scalar_product(sine_part[..., None])       # [cases, times, configurations] Scalar
probabilities = cosine_amplitudes.squared() + sine_amplitudes.squared()            # [cases, times, configurations] Scalar
spin_only = np.sin(exchange_phase / 2) ** 2
render.draw_exchange(exchange_phase, probabilities, spin_only);

In [ ]:
values = probabilities.to_array()                                                  # [cases, times, configurations]
print("total probability, largest deviation from one:", np.abs(values.sum(axis=-1) - 1).max())
print("both electrons on one site at most, per regime:", (values[..., 0] + values[..., 5]).max(axis=-1))
print("swap less the spin-only prediction at most, per regime:", np.abs(values[..., 3] - spin_only).max(axis=-1))

Each configuration's amplitude is a point in a plane: its cosine part across, its sine part up, and its probability the squared distance from the centre. Below, one row per regime and one dial per configuration that can ever be occupied. On the two singly occupied dials the arrow is the triplet's, standing still at zero energy, followed by the singlet's, turning from its tip: where the two add up the spins have not swapped, and where they cancel they have. Without repulsion the singlet's arrows swing back and forth along a line. In the crossover they trace rosettes, the slow swap with the fast charge motion riding on it. At strong repulsion the singlet's arrow circles the triplet's tip, and the swap is that interference alone, while the shared-site dials barely stir.

In [ ]:
def dials(parts):
    """Each configuration's amplitude in the cosine and sine parts, and the share of the triplet of zero
    spin in each."""
    shares = [triplets[1] * triplets[1].reverse().scalar_product(part) for part in parts]   # [..., times] Pair each
    return [configurations.reverse().scalar_product(state[..., None]) for state in (*parts, *shares)]


# Every few steps of the cycle, one row per regime.
stride = 6
frame_ms = 40
rows = [f"{name}\n$U/t = {ratio:g}$" for name, ratio in zip(names, repulsion_ratios)]
render.inline(render.animate_dials(rows, *dials([cosine_part[:, ::stride], sine_part[:, ::stride]])), frame_ms)

## 5. A field that differs between the sites

A field along z that differs between the two sites, as for electrons in two neighbouring quantum dots, is one more one-electron map: it raises spin up and lowers spin down on the left, and the other way round on the right. It lifts to pairs by the same trace as the hopping. Up-down and down-up are states of their own under it, with opposite energies, so it mixes the singlet with the triplet of zero spin, and the block on which the Hamiltonian squared to a number is gone. Instead the Hamiltonian is diagonalized, and each of its states turns by its own energy, the cosine and the sine of the Hamiltonian made of theirs.

Below, at strong repulsion, the field difference against the exchange: none, equal to it, and five times it, over one exchange cycle. The triplet's arrow now turns as well. Without a field difference the spins swap as before. Once the field difference outweighs the exchange, up-down stays up-down: the swap freezes, and the two spins each keep their own state.

In bra-ket notation the field difference reads as $\tfrac{\Delta}{2}\left(\sigma_{Lz} - \sigma_{Rz}\right)$, and at strong repulsion the pair is the singlet–triplet qubit of the [two spins](../two_spins/two_spins.ipynb) notebook, turning about an axis set by the exchange against the field difference.

In [ ]:
orbitals = stack([left_up, left_down, right_up, right_down])                       # [orbitals] Orbital
# Up raised and down lowered on the left, the other way round on the right, by half the difference each.
field_signs = np.array([1.0, -1.0, -1.0, 1.0])
field_difference = 0.5 * (field_signs * (orbitals * (orbitals | Orbital))).sum(axis=0)   # [] Orbital <- Orbital
pair_field = lift(field_difference)                                                 # [] Pair <- Pair


def turn(energies, modes, start, times):
    """The cosine and the sine of the Hamiltonian times time applied to a start, from the Hamiltonian's
    states: each keeps its share of the start and turns by its own energy."""
    shares = modes * modes.reverse().scalar_product(start)                         # [..., modes] Pair
    angles = energies[..., None, :] * times[..., :, None]                          # [..., times, modes] Scalar
    return (shares[..., None, :] * angles.cos()).sum(axis=-1), (shares[..., None, :] * angles.sin()).sum(axis=-1)


# At strong repulsion, field differences against the exchange, one entry per case.
strong, strong_gap = repulsions[-1], gaps[-1]
field_ratios = np.array([0.0, 1.0, 5.0])
field_hamiltonians = hamiltonian(strong) + field_ratios * strong_gap * pair_field   # [cases] Pair <- Pair
field_energies, field_modes = field_hamiltonians.eigh()                             # [cases, modes] Scalar, Pair
field_times = exchange_phase[::stride] / strong_gap                                 # [times]
field_parts = turn(field_energies, field_modes, left_up ^ right_down, field_times)   # [cases, times] Pair each
cosine_field, sine_field = (configurations.reverse().scalar_product(part[..., None]) for part in field_parts)   # [cases, times, configurations] Scalar each
field_probabilities = cosine_field.squared() + sine_field.squared()                # [cases, times, configurations] Scalar
field_rows = [f"field difference\n{ratio:g} exchange" for ratio in field_ratios]
render.inline(render.animate_dials(field_rows, *dials(field_parts)), frame_ms)

In [ ]:
values = field_probabilities.to_array()                                            # [cases, times, configurations]
print("field difference on up-down, less up-down:", np.abs((pair_field(left_up ^ right_down) - (left_up ^ right_down)).kernel).max())
print("total probability, largest deviation from one:", np.abs(values.sum(axis=-1) - 1).max())
print("spins swapped at most, per field difference:", values[..., 3].max(axis=-1))

## 6. One site lower than the other

Lower one site's energy against the other's, the same for both spins, and the electrons are drawn to it. The asymmetry is one more one-electron map, lifted by the same trace, and the lifted map is also what measures the difference in occupation between the sites: the ground state's expectation of it says how much charge has moved over. Without repulsion the charge slides over at once, both electrons ending on the lower site. With strong repulsion the electrons hold one per site, since putting both on the lower site costs the repulsion, until the asymmetry outweighs it; then the charge moves over across a narrow range. Strong correlation and charge transfer compete, and the crossover sits where the asymmetry matches the repulsion.

In the notation of second quantization the asymmetry reads as $\tfrac{\Delta v}{2}\sum_\sigma(\hat n_{R\sigma} - \hat n_{L\sigma})$; the same occupation curve opens the density functional treatment of Carrascal, Ferrer, Smith and Burke.

In [ ]:
# The left site lowered and the right raised, by half the asymmetry each, for both spins alike.
site_signs = np.array([-1.0, -1.0, 1.0, 1.0])
site_difference = lift(0.5 * (site_signs * (orbitals * (orbitals | Orbital))).sum(axis=0))   # [] Pair <- Pair

# Asymmetries up to twice the strongest repulsion, for each regime; the ground state of each.
asymmetries = hopping_strength * np.linspace(0.0, 2 * repulsions[-1], 401)
_, asymmetric_modes = (hamiltonian(repulsions[:, None]) + asymmetries * site_difference).eigh()   # [cases, samples, modes] Pair
lowest = asymmetric_modes[..., 0]                                                   # [cases, samples] Pair
# The left site's occupation less the right's: minus twice the lifted site difference.
transferred = -2 * lowest.reverse().scalar_product(site_difference(lowest))         # [cases, samples] Scalar
render.draw_transfer(asymmetries / hopping_strength, transferred, names, repulsion_ratios);

In [ ]:
at_half, at_twice = np.searchsorted(asymmetries, [0.5 * repulsions[-1], 2 * repulsions[-1]])
print("charge moved at half the strongest repulsion, per regime:", transferred.to_array()[:, at_half])
print("charge moved at twice the strongest repulsion, per regime:", transferred.to_array()[:, at_twice])

In [ ]:
# checks
# Equal-spin hops are blocked and the opposite-spin triplet cancels; the singlet's two routes add up to
# the symmetric double occupancy. On the singlet block the shifted Hamiltonian applied twice is the rate
# squared, and the gap it gives matches the full spectrum; the projected singlet is the ground state.
# Spin up on the left and spin down on the right is half singlet, half triplet, and stays normalized.
# The field difference leaves up-down a state of its own, and a strong one freezes the swap. Strong
# repulsion holds one electron per site until the asymmetry outweighs it.
np.testing.assert_allclose(blocked.kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((singlet_image + 2 * hopping_strength * symmetric_double).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((twice - block_rate(repulsion_sweep[:, None]) ** 2 * block).kernel, 0.0, atol=1e-10)
np.testing.assert_allclose(exchange_gap, energies.to_array()[:, 1] - energies.to_array()[:, 0], atol=1e-10)
np.testing.assert_allclose(ground_energies.to_array(), repulsions / 2 - block_rate(repulsions), atol=1e-12)
np.testing.assert_allclose(((singlet + triplets[1]) / np.sqrt(2) - (left_up ^ right_down)).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose(probabilities.to_array().sum(axis=-1), 1.0, atol=1e-11)
np.testing.assert_allclose((pair_field(left_up ^ right_down) - (left_up ^ right_down)).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose(field_probabilities.to_array().sum(axis=-1), 1.0, atol=1e-11)
assert field_probabilities.to_array()[-1, :, 3].max() < 0.05
assert transferred.to_array()[-1, at_half] < 0.1
assert transferred.to_array()[-1, at_twice] > 1.9